In [ ]:
import timesfm

In [ ]:
tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=4,
          horizon_len=12,
          num_layers=50,
          use_positional_embedding=False,
          context_len=2048,
      ),
      checkpoint=timesfm.TimesFmCheckpoint(
          huggingface_repo_id="google/timesfm-2.0-500m-pytorch"),
  )

In [ ]:
import pickle
import torch
import numpy as np
from tqdm import tqdm

In [ ]:
def masked_mse(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = (preds - labels)**2
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def masked_rmse(preds, labels, null_val):
    return torch.sqrt(masked_mse(preds=preds, labels=labels, null_val=null_val))


def masked_mae(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels)
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def masked_mape(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels) / labels
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def compute_all_metrics(preds, labels, null_val):
    mae = masked_mae(preds, labels, null_val).item()
    mape = masked_mape(preds, labels, null_val).item()
    rmse = masked_rmse(preds, labels, null_val).item()
    return mae, mape, rmse

In [ ]:
def generate_expert_samples(data, idx, past_steps, future_steps, stride):
    """"
    据给定的索引 idx, 从输入数据 data 中生成输入 x 和输出 y 的数据集
    """
    _, N, F = data.shape  # 获取数据的形状
    
    res = data[idx]  # 根据索引获取数据    
    length = len(idx)-1  # 获取索引长度并减1
    x_index, y_index = [], []  # 初始化x和y的索引列表
    
    # 遍历索引生成x和y的索引
    for i in range(length, 0, -stride):
        if i - past_steps - future_steps >= 0:
            x_index.extend(list(range(i - past_steps - future_steps, i - future_steps)))
            y_index.extend(list(range(i - future_steps, i)))
    
    # 将索引转换为数组
    x_index = np.asarray(x_index)
    y_index = np.asarray(y_index)
    
    # 重塑数据
    x = res[x_index].reshape(-1, past_steps, N, F).transpose(0, 3, 2, 1)
    y = res[y_index].reshape(-1, future_steps, N, F).transpose(0, 3, 2, 1)[:, :1, ...]
    
    return x, y

In [ ]:
datasets_list = ['pems03_flow', 'pems04_flow', 'pems07_flow', 'pems08_flow', 'occpairs_occupancy', 'occhamburg_occupancy', 'pemsbay_speed', 'metrla_speed', 'trafficsh_speed', 'bikenyc_inflow', 'taxinyc_inflow', 'tdrive_inflow']
folder_path = "/data/weichen/ST-Library/datasets/eval_datasets"

for dataset in datasets_list:
    
    print('*'*30)
    print(f"Dataset: {dataset}")
    print('*'*30)
    
    # for num_steps in [12, 24]:
    for num_steps in [12]:
    
        with open(f"{folder_path}/{dataset}/{dataset}_temporal.pkl", 'rb') as f:
            df = pickle.load(f)
            raw_temporal_data = torch.tensor(df.values).unsqueeze(-1)
            f.close()

        T, N, _ = raw_temporal_data.shape

        # train_rate = few_shot_ratio if few_shot_ratio <= train_val_test_rate[0] else train_val_test_rate[0]
        test_rate = 0.2
        test_idx = [i for i in range(int(T * (1 - test_rate)), T)]
        
        test_x, test_y = generate_expert_samples(raw_temporal_data.numpy(), test_idx, past_steps=num_steps, future_steps=num_steps, stride=1)
        
        # test_x = test_x[:1]
        # test_y = test_y[:1]
        
        B, F, N, T = test_x.shape
        
        # 使用TimesFM进行预测
        forecast_input = test_x.reshape(B * N, T)
        frequency_input = [0] * B * N
        
        point_forecast, _ = tfm.forecast(
            forecast_input,
            freq=frequency_input,
        )
        preds = point_forecast.reshape(B, N, T)
        labels = test_y.reshape(B, N, T)
        
        labels = torch.Tensor(labels).permute(0, 2, 1)
        preds = torch.Tensor(preds).permute(0, 2, 1)
        
        # handle the precision issue when performing inverse transform to label
        mask_value = torch.tensor(0)

        test_mae = []
        test_mape = []
        test_rmse = []

        # Calculate metrics
        for i in range(num_steps):
            res = compute_all_metrics(preds[:,i,:], labels[:,i,:], mask_value)
            test_mae.append(res[0])
            test_mape.append(res[1] * 100)
            test_rmse.append(res[2])

        mae_mean = np.mean(test_mae)
        mae_std = 0
        rmse_mean = np.mean(test_rmse)
        rmse_std = 0
        mape_mean = np.mean(test_mape)
        mape_std = 0
        
        if num_steps == 12:
            print('TimesFM - Short Forecasting' + "\t\t MAE:" + "& $" + f"{mae_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mae_std:.2f}'+"}}$" + "\t\t RMSE:" + "& $" + f"{rmse_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{rmse_std:.2f}'+"}}$" + "\t\t MAPE:" + "& $" + f"{mape_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mape_std:.2f}'+"}}$")
        else:
            print('TimesFM - Long Forecasting' + "\t\t MAE:" + "& $" + f"{mae_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mae_std:.2f}'+"}}$" + "\t\t RMSE:" + "& $" + f"{rmse_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{rmse_std:.2f}'+"}}$" + "\t\t MAPE:" + "& $" + f"{mape_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mape_std:.2f}'+"}}$")

    #     break
    # break